# Ejercicio 1: ¿deriva de datos o deriva conceptual?

El modelo de la tarifa (la red de 64 neuronas, versión 1) se entrenó con los viajes de septiembre y octubre. En producción, cada semana llegan unos 300 viajes. Este notebook simula dos cosas que pueden pasarle a esa semana y mide, para cada una, dos números: cuánto se parecen los viajes que llegan a los de entrenamiento (el PSI de dos features) y cuánto se equivoca el modelo (el MAE, que solo se conoce cuando llega la tarifa cobrada).

- **Deriva de datos:** cambia lo que llega. Una semana de feriado, con muchos más viajes al aeropuerto.
- **Deriva conceptual:** cambia la relación entre lo que llega y lo que hay que predecir. La ciudad aumenta 20% la tarifa por taxímetro.

Corre todas las celdas antes de tocar nada (**Run All**): corre de principio a fin sin los `TODO` resueltos.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from preparacion_taxis import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
modelo = Pipeline([
    ("preparar", preparacion()),
    ("modelo", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42)),
]).fit(X_ent, y_ent)

# Los viajes de prueba, con su tarifa: de aqui salen las semanas simuladas.
prueba = viajes.loc[X_prueba.index].copy()
COLUMNAS = NUMERICAS + YA_LISTAS + CATEGORICAS
print("MAE en prueba:", round(mean_absolute_error(y_prueba, modelo.predict(X_prueba)), 4))

MAE en prueba: 1.1695


## Paso 1: el PSI

El índice de estabilidad de la población (PSI) compara dos distribuciones repartidas en las mismas casillas. Si en la casilla $i$ la referencia tiene la proporción $r_i$ y la semana actual $a_i$:

$$PSI = \sum_i (a_i - r_i) \ln\frac{a_i}{r_i}$$

Es 0 si las dos son iguales, y crece cuanto más se separan. Para una feature numérica, las casillas son los deciles de la referencia (10 casillas con 10% de los viajes de entrenamiento cada una); para una categórica, cada categoría es una casilla. Una casilla vacía haría $\ln 0$: se reemplaza por una proporción mínima de 0.0001.

In [2]:
# Las casillas de distancia_km: los deciles de entrenamiento, abiertas en los extremos.
CORTES = np.quantile(X_ent["distancia_km"].dropna(), np.linspace(0, 1, 11))
CORTES[0], CORTES[-1] = -np.inf, np.inf
ZONAS = sorted(X_ent["zona_origen"].unique())


def proporciones_numericas(valores, cortes):
    conteo, _ = np.histogram(valores.dropna(), bins=cortes)
    return conteo / conteo.sum()


def proporciones_categoricas(valores, categorias):
    return valores.value_counts(normalize=True).reindex(categorias, fill_value=0).to_numpy()


def psi(referencia, actual, minimo=0.0001):
    # DONE 1: reemplaza las proporciones menores que `minimo` por `minimo`
    # (np.clip) en las dos, y devuelve la suma de la formula de arriba.
    r = np.clip(np.asarray(referencia, dtype=float), minimo, None)
    a = np.clip(np.asarray(actual, dtype=float), minimo, None)
    return float(np.sum((a - r) * np.log(a / r)))


REF_DISTANCIA = proporciones_numericas(X_ent["distancia_km"], CORTES)
REF_ZONA = proporciones_categoricas(X_ent["zona_origen"], ZONAS)


def medir(semana):
    """PSI de las dos features y MAE del modelo en una semana de viajes."""
    return {
        "psi_distancia": round(psi(REF_DISTANCIA, proporciones_numericas(semana["distancia_km"], CORTES)), 4),
        "psi_zona": round(psi(REF_ZONA, proporciones_categoricas(semana["zona_origen"], ZONAS)), 4),
        "mae": round(mean_absolute_error(semana[OBJETIVO], modelo.predict(semana[COLUMNAS])), 4),
    }


print("La prueba completa contra entrenamiento:", medir(prueba))

La prueba completa contra entrenamiento: {'psi_distancia': 0.0174, 'psi_zona': 0.0089, 'mae': 1.1695}


## Paso 2: cuánto se mueve una semana normal

Una semana de 300 viajes nunca es idéntica a la referencia, aunque nada haya cambiado: el PSI de una semana normal no es 0. Antes de decidir desde cuánto alertar, hay que saber cuánto se mueve solo por azar. `semana_normal` toma 300 viajes al azar de la prueba.

In [3]:
def semana_normal(semilla):
    return prueba.sample(300, replace=True, random_state=semilla)


# DONE 2: mide 200 semanas normales (semillas 0 a 199) y guarda los
# diccionarios de medir() en una lista; despues, un DataFrame.
normales = pd.DataFrame([medir(semana_normal(s)) for s in range(200)])

print("Semanas normales:", len(normales))
print(normales.describe(percentiles=[0.95]).loc[["mean", "95%", "max"]].round(4))

Semanas normales: 200
      psi_distancia  psi_zona     mae
mean         0.0497    0.0354  1.1639
95%          0.0923    0.0795  1.2672
max          0.1214    0.1091  1.3164


## Paso 3: dos derivas simuladas

`semana_de_feriado` toma 300 viajes como `semana_normal`, pero los que van o vienen del aeropuerto tienen `peso` veces más probabilidad de salir. `semana_con_aumento` toma una semana normal y multiplica la tarifa de los viajes sin tarifa fija por `aumento`: los viajes son los mismos, lo que cambia es lo que se cobra.

In [4]:
def semana_de_feriado(semilla, peso):
    del_aeropuerto = (prueba["zona_origen"] == "JFK Airport") | (prueba["zona_destino"] == "JFK Airport")
    # DONE 3: pesos de 1 para cada viaje, y `peso` para los del aeropuerto
    # (np.where), y prueba.sample(300, replace=True, weights=..., random_state=semilla).
    pesos = np.where(del_aeropuerto, peso, 1)
    return prueba.sample(300, replace=True, weights=pesos, random_state=semilla)


def semana_con_aumento(semilla, aumento):
    semana = semana_normal(semilla).copy()
    # DONE 4: multiplica OBJETIVO por `aumento` en las filas con es_tarifa_fija == 0.
    semana.loc[semana["es_tarifa_fija"] == 0, OBJETIVO] *= aumento
    return semana


print("Feriado, peso 4:     ", medir(semana_de_feriado(1, 4)))
print("Aumento de 20%:      ", medir(semana_con_aumento(1, 1.20)))

Feriado, peso 4:      {'psi_distancia': 0.1887, 'psi_zona': 0.1964, 'mae': 0.9694}
Aumento de 20%:       {'psi_distancia': 0.0767, 'psi_zona': 0.0615, 'mae': 4.1638}


## Paso 4: cómo crece cada número con la intensidad

Diez semanas por cada intensidad, y el promedio de cada medida.

In [5]:
filas = []
for peso in [1, 2, 4, 8]:
    medidas = pd.DataFrame([medir(semana_de_feriado(s, peso)) for s in range(10)]).mean()
    filas.append({"deriva": "datos (aeropuerto x" + str(peso) + ")", **medidas.round(4).to_dict()})
for aumento in [1.0, 1.05, 1.10, 1.20]:
    medidas = pd.DataFrame([medir(semana_con_aumento(s, aumento)) for s in range(10)]).mean()
    filas.append({"deriva": "conceptual (+" + str(round((aumento - 1) * 100)) + "%)", **medidas.round(4).to_dict()})
pd.DataFrame(filas)

,deriva,psi_distancia,psi_zona,mae
0,datos (aeropuerto x1),0.0510,0.0326,1.1919
1,datos (aeropuerto x2),0.0591,0.0553,1.1548
2,datos (aeropuerto x4),0.2211,0.1965,1.0441
3,datos (aeropuerto x8),0.5422,0.4193,0.9440
4,conceptual (+0%),0.0456,0.0426,1.1919
5,conceptual (+5%),0.0456,0.0426,1.5333
6,conceptual (+10%),0.0456,0.0426,2.2545
7,conceptual (+20%),0.0456,0.0426,4.0505


## Paso 5: el umbral

Una alerta de deriva de datos dispara cuando el PSI de alguna de las dos features pasa un umbral. Un umbral bajo detecta antes y da falsas alarmas en semanas normales; uno alto, al revés. La tabla cuenta, para cada umbral candidato, qué parte de las 200 semanas normales alertaría (falsas alarmas) y qué parte de 200 semanas de feriado con peso 4 alertaría (detección).

In [6]:
feriados = pd.DataFrame([medir(semana_de_feriado(s, 4)) for s in range(200)])


def alerta(medidas, umbral):
    return (medidas["psi_distancia"] > umbral) | (medidas["psi_zona"] > umbral)


tabla = pd.DataFrame([
    {"umbral": u, "falsas_alarmas": alerta(normales, u).mean(), "deteccion_feriado_x4": alerta(feriados, u).mean()}
    for u in [0.05, 0.10, 0.15, 0.20, 0.25]
])
tabla

,umbral,falsas_alarmas,deteccion_feriado_x4
0,0.05,0.495,1.000
1,0.10,0.040,1.000
2,0.15,0.000,0.975
3,0.20,0.000,0.715
4,0.25,0.000,0.320


## Respuestas

Responde en el README del laboratorio (Respuestas 1 a 4).